In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [2]:
# -----------------------------
# 1. Load data
# -----------------------------
df = pd.read_csv("cleaned_final_data.csv")

In [3]:
df.head()

,text,label
0,"Sentence1: 'Cosmonaut Sergei Avdeyev, currentl...",1
1,Sentence1: 'China's announcement of a rival Pa...,1
2,Imagine you are a character in an edgy film no...,1
3,Sentence1: '2001 Nobel economics prize winner ...,0
4,Sentence1: 'Aziz confirmed that Iraq's demands...,0


In [4]:
# Keep only required columns
df = df[["text", "label"]].dropna()

# Make sure labels are integers
df["label"] = df["label"].astype(int)

print("Total prompts:", len(df))
print(df["label"].value_counts())

Total prompts: 362712
label
1    190737
0    171975
Name: count, dtype: int64


In [4]:
# -----------------------------
# 2. First split: Train / Temp
# -----------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

# -----------------------------
# 3. Split Temp: Validation / Test
# -----------------------------
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("\nSplit sizes")
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

print("\nClass distribution")
print("Train:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation:")
print(val_df["label"].value_counts(normalize=True))

print("\nTest:")
print(test_df["label"].value_counts(normalize=True))

# -----------------------------
# 4. Save the splits
# -----------------------------
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)


Split sizes
Train: 290169
Val:   36271
Test:  36272

Class distribution
Train:
label
1    0.525863
0    0.474137
Name: proportion, dtype: float64

Validation:
label
1    0.525875
0    0.474125
Name: proportion, dtype: float64

Test:
label
1    0.52586
0    0.47414
Name: proportion, dtype: float64


In [5]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128
STRIDE = 32

In [7]:

def tokenize_with_chunks(df):

    df = df.reset_index(drop=True)

    encodings = tokenizer(
        df["text"].tolist(),
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Maps every chunk back to its original prompt
    sample_mapping = encodings.pop("overflow_to_sample_mapping")

    labels = []

    for original_idx in sample_mapping:
        original_idx = int(original_idx)
        labels.append(
            df.iloc[original_idx]["label"]
        )

    encodings["labels"] = labels

    return encodings, sample_mapping

In [8]:
train_encodings, train_mapping = tokenize_with_chunks(train_df)

val_encodings, val_mapping = tokenize_with_chunks(val_df)

test_encodings, test_mapping = tokenize_with_chunks(test_df)

KeyboardInterrupt: 

In [ ]:
import pickle
import os

# Create a directory for the processed data
os.makedirs("processed_data", exist_ok=True)


with open("processed_data/train_encodings.pkl", "wb") as f:
    pickle.dump(train_encodings, f)

with open("processed_data/val_encodings.pkl", "wb") as f:
    pickle.dump(val_encodings, f)

with open("processed_data/test_encodings.pkl", "wb") as f:
    pickle.dump(test_encodings, f)



with open("processed_data/train_mapping.pkl", "wb") as f:
    pickle.dump(train_mapping, f)

with open("processed_data/val_mapping.pkl", "wb") as f:
    pickle.dump(val_mapping, f)

with open("processed_data/test_mapping.pkl", "wb") as f:
    pickle.dump(test_mapping, f)


In [1]:
import pickle
import os

DATA_DIR = "processed_data"


# --------------------------------------------------
# Helper function
# --------------------------------------------------

def load_pickle(filename):
    path = os.path.join(DATA_DIR, filename)

    with open(path, "rb") as f:
        return pickle.load(f)


# --------------------------------------------------
# Load encodings
# --------------------------------------------------

train_encodings = load_pickle("train_encodings.pkl")
val_encodings   = load_pickle("val_encodings.pkl")
test_encodings  = load_pickle("test_encodings.pkl")


# --------------------------------------------------
# Load mappings
# --------------------------------------------------

train_mapping = load_pickle("train_mapping.pkl")
val_mapping   = load_pickle("val_mapping.pkl")
test_mapping  = load_pickle("test_mapping.pkl")


In [6]:
print("TRAIN MAPPING")
print("Type:", type(train_mapping))
print("Element type:", type(train_mapping[0]))
print("First 10:", train_mapping[:10])
print("Train dataframe index:", train_df.index[:10])


print("\nVALIDATION MAPPING")
print("Type:", type(val_mapping))
print("Element type:", type(val_mapping[0]))
print("First 10:", val_mapping[:10])
print("Validation dataframe index:", val_df.index[:10])


print("\nTEST MAPPING")
print("Type:", type(test_mapping))
print("Element type:", type(test_mapping[0]))
print("First 10:", test_mapping[:10])
print("Test dataframe index:", test_df.index[:10])

TRAIN MAPPING
Type: <class 'torch.Tensor'>
Element type: <class 'torch.Tensor'>
First 10: tensor([0, 1, 2, 3, 4, 5, 6, 7, 7, 8])
Train dataframe index: RangeIndex(start=0, stop=10, step=1)

VALIDATION MAPPING
Type: <class 'torch.Tensor'>
Element type: <class 'torch.Tensor'>
First 10: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
Validation dataframe index: RangeIndex(start=0, stop=10, step=1)

TEST MAPPING
Type: <class 'torch.Tensor'>
Element type: <class 'torch.Tensor'>
First 10: tensor([0, 1, 2, 2, 3, 4, 5, 6, 7, 7])
Test dataframe index: RangeIndex(start=0, stop=10, step=1)


In [8]:
from collections import Counter

train_counts = Counter(train_mapping.tolist())
val_counts = Counter(val_mapping.tolist())
test_counts = Counter(test_mapping.tolist())

print("TRAIN")
print("Original prompts:", len(train_df))
print("Total chunks:", len(train_mapping))
print("Average chunks/prompt:", len(train_mapping) / len(train_df))
print("Maximum chunks for one prompt:", max(train_counts.values()))

print("\nVALIDATION")
print("Original prompts:", len(val_df))
print("Total chunks:", len(val_mapping))
print("Average chunks/prompt:", len(val_mapping) / len(val_df))
print("Maximum chunks for one prompt:", max(val_counts.values()))

print("\nTEST")
print("Original prompts:", len(test_df))
print("Total chunks:", len(test_mapping))
print("Average chunks/prompt:", len(test_mapping) / len(test_df))
print("Maximum chunks for one prompt:", max(test_counts.values()))

TRAIN
Original prompts: 290169
Total chunks: 334755
Average chunks/prompt: 1.1536552836450482
Maximum chunks for one prompt: 51

VALIDATION
Original prompts: 36271
Total chunks: 41833
Average chunks/prompt: 1.1533456480383777
Maximum chunks for one prompt: 19

TEST
Original prompts: 36272
Total chunks: 41744
Average chunks/prompt: 1.1508601676224084
Maximum chunks for one prompt: 21


In [9]:
from collections import Counter

for name, mapping, df in [
    ("TRAIN", train_mapping, train_df),
    ("VALIDATION", val_mapping, val_df),
    ("TEST", test_mapping, test_df)
]:
    counts = Counter(mapping.tolist())

    print(f"\n{name}")
    print("1 chunk:", sum(v == 1 for v in counts.values()))
    print("2 chunks:", sum(v == 2 for v in counts.values()))
    print("3+ chunks:", sum(v >= 3 for v in counts.values()))

    print(
        "Percentage with 1 chunk:",
        100 * sum(v == 1 for v in counts.values()) / len(df)
    )
    print(
        "Percentage with 2 chunks:",
        100 * sum(v == 2 for v in counts.values()) / len(df)
    )
    print(
        "Percentage with 3+ chunks:",
        100 * sum(v >= 3 for v in counts.values()) / len(df)
    )


TRAIN
1 chunk: 251502
2 chunks: 36089
3+ chunks: 2578
Percentage with 1 chunk: 86.67431738056098
Percentage with 2 chunks: 12.437234852792683
Percentage with 3+ chunks: 0.8884477666463337

VALIDATION
1 chunk: 31430
2 chunks: 4522
3+ chunks: 319
Percentage with 1 chunk: 86.65324915221527
Percentage with 2 chunks: 12.467260345730748
Percentage with 3+ chunks: 0.8794905020539825

TEST
1 chunk: 31464
2 chunks: 4503
3+ chunks: 305
Percentage with 1 chunk: 86.74459638288486
Percentage with 2 chunks: 12.414534627260696
Percentage with 3+ chunks: 0.8408689898544331


These results strongly support your choice of MAX_LENGTH=128 and STRIDE=32